<a href="https://colab.research.google.com/github/somendrew/RAG_System/blob/main/MultiDoc_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📘 Notes: Multi-document RAG

### The Current Retrieval Problem
Our current retriever searches across 'Solar System', 'Earth', and 'Moon' chunks mixed in one Chroma collection. While technically multi-document, it lacks:

*   **Scoped Questions:** No way to ask questions confined to a single document (e.g., "Only look at the Moon page").
*   **Source Attribution:** Inability to identify the source document for an answer.
*   **Conflict Resolution:** No mechanism to handle genuinely conflicting information from different documents (e.g., Doc A says X, Doc B says Y).

### What "Multi-document RAG" Adds (Conceptually)

Multi-document RAG enhances retrieval with the following key features:

*   **Metadata Tagging at Index Time:** Each chunk carries `metadata={"title": ..., "source": ...}`. This metadata is crucial for subsequent operations.
*   **Metadata Filtering at Retrieval Time:** Instead of searching all chunks, we can restrict the search to chunks based on metadata (e.g., `metadata["title"] == "Moon"`). Chroma supports this natively via `search_kwargs={"filter": {...}}`.
*   **Source Attribution in the Answer:** Retrieved chunks retain their `metadata["title"]`, allowing the LLM to cite sources or for us to display them alongside the answer.
*   **(Optional, More Advanced) Query Routing:** An LLM decides which document(s) are relevant to a question *before* retrieval. For example, a question like "Does Earth have seasons?" could be routed exclusively to Earth-related documents, rather than searching all three. This is a preview of "adaptive" RAG.

### Where This Sits Relative to Adaptive/Agentic RAG

Routing in Multi-document RAG introduces an "adaptive" idea, where the system makes a retrieval decision based on the question, rather than always using a fixed retrieval method. For now, we'll keep the routing simple (LLM selects from a known list of documents) and explore more complex routing logic later.

### LLM Router — How It Works

Before retrieval, an additional LLM call is made (similar to the query-rewrite step in Conversational RAG). This LLM is presented with the question and a list of available document titles, then asked to select the relevant one(s). This selection is then used as a metadata filter for Chroma.

**Process Flow:**

`Question` → `[LLM Router: picks title]` → `Filtered retrieval` → `Answer`

This structure is intentionally similar to Conversational RAG's rewrite step, where an LLM's sole purpose is to determine how to retrieve information before the actual retrieval takes place.

# Block Diagram

```text
┌─────────────────────────────────────────────────────────────────┐
│                     INDEXING PHASE                              │
│                 (Runs once, offline)                            │
└─────────────────────────────────────────────────────────────────┘

   "Solar System"             "Earth"                  "Moon"
    topic string            topic string            topic string
         │                       │                       │
         └───────────────────────┼───────────────────────┘
                                 │
                                 ▼
                      fetch_wikipedia_page()
                 Calls Wikipedia REST API per topic
                                 │
                                 ▼
                     docs: list of 3 Documents
                    page_content + metadata.title
                                 │
                                 ▼
                    RecursiveCharacterTextSplitter
                      chunk_size 1000, overlap 150
                                 │
                                 ▼
                        chunks: many Documents
                 each keeps parent's title metadata
                                 │
                                 ▼
                          OpenAIEmbeddings
            text-embedding-3-small, each chunk to vector
                                 │
                                 ▼
                      Chroma.from_documents()
            vectorstore: vectors + text + title metadata
           stored together, ready for filtered search
                                 │
                                 │
                                 │ (Vectorstore Ready)
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────┐
│                       QUERY PHASE                               │
│         (Runs per question, inside multi_doc_rag)               │
└─────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
                           question: str
                  "Why does the Moon show phases?"
                                 │
                                 ▼
                       router_chain.invoke()
             router_prompt | llm | PydanticOutputParser
                            LLM call #1
                                 │
                                 ▼
                     RouteQuery(title="Moon")
                   structured Pydantic object
                                 │
                                 ▼
                 as_retriever(filter={title: Moon})
                 scopes search to Moon chunks only
                                 │
                                 ▼
                filtered_retriever.invoke(question)
                      similarity search, k=4
                                 │
                                 ▼
                       docs: 4 Moon chunks
                  most relevant to the question
                                 │
                                 ▼
                           format_docs()
                joins chunks into one context string
                                 │
                                 ▼
                       answer_chain.invoke()
             answer_prompt | llm | StrOutputParser
             LLM call #2, answer grounded in context

```

### Install Dependencies

In [1]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma chromadb tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

### Load API Key

In [2]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('api_key')

### Data Ingestion: Fetch Wikipedia Pages

In [3]:

import requests
from langchain_core.documents import Document

def fetch_wikipedia_page(title: str) -> Document:
    """
    Fetches a Wikipedia page's extract as a LangChain Document.

    Args:
        title (str): The title of the Wikipedia page to fetch.

    Returns:
        Document: A LangChain Document object containing the page content and metadata.
    """
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",       # Action to perform: query the API.
        "prop": "extracts",      # Request page extracts.
        "explaintext": True,     # Return plain text instead of HTML.
        "titles": title,         # The title of the page to query.
        "format": "json",        # Response format: JSON.
        "redirects": 1,          # Resolve redirects.
    }
    headers = {"User-Agent": "RAG-Learning-Notebook/1.0 (educational use)"}

    resp = requests.get(url, params=params, headers=headers, timeout=10)
    resp.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx).
    data = resp.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))
    text = page.get("extract", "")

    return Document(page_content=text, metadata={"title": page.get("title", title), "source": url})

topics = ["Solar System","Moon"]

docs = [fetch_wikipedia_page(t) for t in topics]

for d in docs:
    print(d.metadata, "-", len(d.page_content), "chars")

{'title': 'Solar System', 'source': 'https://en.wikipedia.org/w/api.php'} - 61764 chars
{'title': 'Moon', 'source': 'https://en.wikipedia.org/w/api.php'} - 89899 chars


### Text Preprocessing: Chunking Documents

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 150)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

# confirm metadata survived splitting
print(chunks[0].metadata)

Split into 251 chunks
{'title': 'Solar System', 'source': 'https://en.wikipedia.org/w/api.php'}


### Vector Store Creation

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings  = OpenAIEmbeddings(model = "text-embedding-3-small",api_key=OPENAI_API_KEY)

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    collection_name = "multi_doc_rag",
)

### Initialize Language Model (LLM)

In [6]:
from langchain_openai import ChatOpenAI

llm  = ChatOpenAI(model = 'gpt-4o-mini',api_key =   OPENAI_API_KEY)

# Multi-doc RAG Layer
### STEP 7 : Define the Router

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

available_titles = ["Solar System", "Earth", "Moon"]

class RouteQuery(BaseModel):
    title: str = Field(description=f"Which document to search. Must be exactly one of: {available_titles}")

router_parser = PydanticOutputParser(pydantic_object=RouteQuery)

router_prompt = ChatPromptTemplate.from_template(
    "Given the user question, decide which single document is most relevant.\n"
    "Available documents: {titles}\n\n"
    "Question: {question}\n\n"
    "{format_instructions}"
)

router_chain = router_prompt | llm | router_parser

### Define Router and Retrieval Function

In [8]:
def route_and_retrieve(question: str, k: int = 4):
    route = router_chain.invoke({
        "question": question,
        "titles": available_titles,
        "format_instructions": router_parser.get_format_instructions(),
    })

    chosen_title = route.title
    print(f"[Router chose: {chosen_title}]")

    filtered_retriever = vectorstore.as_retriever(
        search_kwargs={"k": k, "filter": {"title": chosen_title}}
    )
    return filtered_retriever.invoke(question), chosen_title

### Answer Generation and Multi-Document RAG Function

In [11]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

answer_prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY this context. Say so if you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

answer_chain = answer_prompt | llm | StrOutputParser()

def multi_doc_rag(question: str):
    docs, chosen_title = route_and_retrieve(question)
    context = format_docs(docs)
    answer = answer_chain.invoke({"context": context, "question": question})
    return {"answer": answer, "source_document": chosen_title, "chunks_used": len(docs)}

### Demonstration / Testing

In [12]:
result1 = multi_doc_rag("Why does the Moon show different phases?")
print(result1["answer"])
print("(source:", result1["source_document"], ")\n")

result2 = multi_doc_rag("How many planets orbit the Sun?")
print(result2["answer"])
print("(source:", result2["source_document"], ")\n")

result3 = multi_doc_rag("What causes seasons on Earth?")
print(result3["answer"])
print("(source:", result3["source_document"], ")\n")

[Router chose: Moon]
The Moon shows different phases due to its rotation and orbit around Earth. As the Moon orbits Earth, its orientation toward the Sun changes, causing varying illumination of its surface. This changing position of sunlight is observable from Earth as the different lunar phases, with the waxing crescent phase representing sunrise and the waning crescent phase representing sunset from our perspective.
(source: Moon )

[Router chose: Solar System]
There are eight planets that orbit the Sun.
(source: Solar System )

[Router chose: Earth]
I don't know.
(source: Earth )

